**NOTE** `phimats` environment should be used as kernel

In [ ]:
import numpy as np

from PreProcessing import PhysicsConfig, MeshConfig, PreProcessing as PP

from MeshManager import MeshManager

from BoundaryConditions import *

from PostProcessing import WriteXDMF

%load_ext autoreload
%autoreload 2

### Simulation data

In [20]:
SimulName = "NRB"
# Element sets
nElementSets = 1
# Number of steps to achieve the load
nSteps = 300

### Read mesh file

In [ ]:
# Element name
elementName = "quad"  		# meshio compatible element name
mesh = MeshManager("NRB.msh", elementName)
mesh.WriteMesh("NRB")

In [22]:
# Create the config object
meshConfig = MeshConfig(
    nTotNodes=mesh.get_nTotNodes(),
    nTotElements=mesh.get_nTotElements(),
    nDim=mesh.get_nDim(),
    materialNames=mesh.getMaterialNames(),
)

### Apply mechanics boundary conditions
**NOTE** This is the total load to be achieved in `nSteps`.

In [ ]:
# Dirichlet BCs list
presBCs = []

# Top pull
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Top"), dof=1, value=1.0e-3))

# Bottom fix
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Bottom"), dof=1, value=0.0))

# Symmetry
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Symmetry_Axis"), dof=0, value=0.0))

# Bottom left corner pin (Fixed in X)
presBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("Fix"), dof=0, value=0.0))

# Write external 
WriteBCVTK(SimulName, mesh, presBCs, dofNames=['ux', 'uy'])

### Mechanical material data

In [ ]:
# Analysis type
AnalysisType = "AxiSymmetricPFF"
# Isotropy
Isotropy = "Isotropic"
# Young's modulus
Emod = 200e9
# Poisson's ratio
nu = 0.3

# Plasticity type
Plasticity = "IsoHard"
HardeningLaw = "PowerLaw"
sig_y0 = 500e6    # Pa
K_hard = 300e6  # Pa
n_pow = 0.3
# Dislocation evolution (if you need these parameters, otherwise, set them to zero)
rho_0 = 1e10  # Initial dislocation density (m⁻²)
M = 3 		  # Taylor factor
alpha = 0.3   # Dislocation interaction constant
b = 2.5e-10   # Burgers vector (m)
G = Emod/(2*(1+nu)) # Shear modulus
k1 = 2*(G/(200*G*alpha*b))  # Multiplication coefficient
k2 = 15  # Recovery coefficient

MechMaterial = {
	"NRB" : {
		"Elastic" : {
			"AnalysisType" : AnalysisType,
			"Isotropy" 	 : Isotropy,
			"Emod" 		 : Emod,
			"nu"   		 : nu,
		},
		"Plastic" : {
			"Plasticity"   : Plasticity,
			"HardeningLaw" : HardeningLaw,
			"sig_y0"       : sig_y0,
			"K_hard"       : K_hard,
			"n_pow"        : n_pow,
   			"rho_0"        : rho_0,
			"M"            : M,
			"b"            : b,
			"alpha"        : alpha,
			"k1"           : k1,
   			"k2"           : k2,
		},
	},
}

MechMaterial["NRB"]["Plastic"]

### PFF material data

In [ ]:
const_ell = 3e-5     # Length-scale regularization parameter [m]
wc = 100e6
# Critical work density [J/m³]
Gc = 2*const_ell*wc	 # Fracture toughness [J/m²]

PFFMaterial = {
	"NRB" : {
		"PFF" : {
			"const_ell" : const_ell,
			"wc" 	    : wc,
		}
    }
}

print("l: ", const_ell, " m")
print("Gc: ", Gc, " J/m²")
print("wc: ", wc, " J/m³")

### Initialize simulation object

### Mechanical

In [26]:
# Create the config object
mechPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "Mechanical",
    PhysicsCategory = "Plastic",
    nSteps    = nSteps,
    presBCs=presBCs
)

In [ ]:
MechNotch = PP(mechPhysConfig, meshConfig, MechMaterial)

In [ ]:
MechNotch.WriteInputFile()

#### PFF

In [29]:
# Create the config object
pffPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "PFF",
    PhysicsCategory = list(PFFMaterial["NRB"].keys())[0],
    nSteps    = nSteps,
)

In [ ]:
PFFNotch = PP(pffPhysConfig, meshConfig, PFFMaterial)

In [ ]:
PFFNotch.WriteInputFile()

### Write output files

In [ ]:
OVERWRITE = False

In [ ]:
MechNotch.WriteOutputFile(overwrite=OVERWRITE)
PFFNotch.WriteOutputFile(overwrite=OVERWRITE)

The four components of `quadAxi` are:

1. **Radial Component ($r$):** Index `0`.
2. **Axial Component ($z$):** Index `1`.
3. **Hoop Component ($\theta$):** Acts in the circumferential direction. This is the "out-of-plane" normal stress/strain that arises due to the change in radius ($u_r/r$). This is index `2`.
4. **Shear Component ($rz$):** Represents the shear in the  plane. In your code, this is index `3`.

In [ ]:

WriteXDMF(SimulName, # Base simulation name
          "NRB",  # Mesh file name
          "quadAxi", # Element name
          nSteps+1, # Number of steps 
          components=["mech", "pff"],
    nDim=2,
    mechModel="Plastic", # Adds strain_p, stress_eq, etc.
    FLUX=True)

**PETSc command line options**
```
./NRB -snes_linesearch_type bt -snes_linesearch_damping 0.8 -snes_linesearch_monitor -snes_linesearch_max_it 50 -snes_max_it 100
```